# 01 — Exploración de datos de celulares

**Pregunta:** ¿qué especificaciones se asocian con las cuatro gamas de precio?

EDA significa análisis exploratorio de datos. Primero comprobamos la calidad;
después exploramos exclusivamente las 1.600 filas de entrenamiento. Las 400 de
test se reservan para el final. Ejecute las celdas en orden con el entorno del proyecto.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "README.profe").exists():
    ROOT = ROOT.parent
if not (ROOT / "README.profe").exists():
    raise RuntimeError("Abra Jupyter desde la raíz del proyecto")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.dataset import load_data, split_data

df = load_data()
X_train, X_test, y_train, y_test = split_data(df)
print("Entrenamiento:", X_train.shape, "Test reservado:", X_test.shape)

## 1. Auditoría de estructura
Contar nulos o clases no se utiliza para ajustar modelos.

In [ ]:
print("Dimensiones:", df.shape)
print("Duplicados:", df.duplicated().sum())
display(pd.DataFrame({"tipo": df.dtypes.astype(str), "nulos": df.isna().sum()}))
display(df.price_range.value_counts().sort_index().rename("filas"))

## 2. Distribuciones del entrenamiento
Los códigos de gama tienen orden, pero no son precios monetarios.

In [ ]:
train = X_train.assign(price_range=y_train)
display(X_train.describe().round(2))
fig, ax = plt.subplots(figsize=(6, 3))
y_train.value_counts().sort_index().plot.bar(ax=ax, color="#267a9e")
ax.set(xlabel="Gama", ylabel="Celulares", title="Clases en entrenamiento")
plt.tight_layout()
plt.show()

## 3. Valores que merecen revisión
Un cero en alto de píxeles o ancho físico no describe una pantalla real. Se conserva el dato original y se documenta; no lo sustituimos arbitrariamente.

In [ ]:
display((X_train[["px_height", "sc_w"]] == 0).sum().rename("ceros en entrenamiento"))

## 4. Correlación
Spearman resume asociaciones monotónicas con el orden de las gamas. No demuestra causalidad ni reemplaza la importancia predictiva del modelo.

In [ ]:
correlation = train.corr(method="spearman")["price_range"].drop("price_range")
display(correlation.sort_values(ascending=False).round(3).rename("Spearman"))
fig, ax = plt.subplots(figsize=(8, 6))
correlation.sort_values().plot.barh(ax=ax, color="#267a9e")
ax.set(title="Asociación con la gama — solo entrenamiento", xlabel="Spearman")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for column, ax in zip(["ram", "battery_power", "px_width"], axes):
    sns.boxplot(data=train, x="price_range", y=column, ax=ax)
plt.tight_layout()
plt.show()

## 5. Decisiones para modelar

- Conservar inicialmente las 20 variables y las filas originales.
- No aplicar sobremuestreo: las clases están equilibradas.
- Escalar para regresión logística y SVM dentro de cada fold; los árboles no lo requieren.
- Contrastar la hipótesis de RAM leyendo la tabla anterior; una asociación fuerte no significa que defina por sí sola el precio.
- No hay fechas: el monitoreo compara particiones reales del dataset como demostración de sesgo de selección, no como vigilancia temporal de producción.

**Para discutir en grupo:** ¿qué variables tienen asociación débil? ¿Puede una variable
con poca correlación ser útil al combinarla con otras? ¿Qué limita trasladar este dataset al mercado actual?
